# WarpTracer — vehicle-mounted LiDAR

One box car, one floor, four walls, with LiDAR mounted above the car.
Run the same acceleration/braking, circle, and S-turn checks while recording scans.
Yellow points show returns; green lines show a sparse selection of laser rays.
All 1080 beams are saved even though the replay displays a subset.

Run the three cells below. Setup reuses `/content/WarpTracer-rigidbody` and its
single root uv environment, switches to `vehicle-lidar`, and updates that branch.
CPU is supported; a GPU runtime uses CUDA when available. The first run compiles
Warp kernels. Orbit, zoom, pause, and scrub the recorded replay.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/WarpTracer-rigidbody")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch",
                    "vehicle-lidar", "https://github.com/yuywe/WarpTracer.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "fetch", "origin", "vehicle-lidar:refs/remotes/origin/vehicle-lidar"], check=True)
if subprocess.run(["git", "show-ref", "--verify", "--quiet", "refs/heads/vehicle-lidar"]).returncode:
    subprocess.run(["git", "branch", "--no-track", "vehicle-lidar", "origin/vehicle-lidar"], check=True)
subprocess.run(["git", "switch", "vehicle-lidar"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "vehicle-lidar"], check=True)
os.environ["UV_PYTHON"] = sys.executable
subprocess.run(["uv", "sync", "--locked", "--extra", "viz"], check=True)


## Choose a driving check

- `"accelerate-brake"`: accelerate straight, then brake to a stop.
- `"circle"`: drive with a constant left steering angle.
- `"s-turn"`: alternate left/right steering, then stop.

Change the scenario and rerun the final two cells. The existing `"drop"` and
`"wall-impact"` checks are also available. Physics runs at 240 Hz; the viewer
records 30 poses per second. Tire support currently assumes a flat floor.

LiDAR scans at 30 Hz with a 270-degree field of view and a 30 m maximum range.
The car follows the same scripted commands; LiDAR does not steer it yet.
Range arrays, validity masks, timestamps, and sensor poses are saved in the NPZ.


In [ ]:
scenario = "accelerate-brake"  # "circle", "s-turn", "drop", or "wall-impact"
subprocess.run(["uv", "run", "--locked", "--extra", "viz", "warptracer-demo",
                "--scenario", scenario], check=True)


In [ ]:
from html import escape
from IPython.display import HTML, display

replay = Path("outputs") / f"{scenario}.html"
display(HTML(f'<iframe srcdoc="{escape(replay.read_text(), quote=True)}" '
             'width="100%" height="650" style="border:0"></iframe>'))
